## Parametric Methods

### Question 1

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# Load data
df = pd.read_csv("gold_silver.csv")

x = df["Year"].values
y = df["GoldPrice_Rs_per_10g"].values

# Manual Linear Regression                                                                                          
xm = np.mean(x)
ym = np.mean(y)

b1 = np.sum((x-xm)*(y-ym)) / np.sum((x-xm)**2)
b0 = ym - b1*xm

# Prediction
yp = b0 + b1*x

# Error, MSE, RMSE
error = y - yp
mse = np.mean(error**2)
rmse = np.sqrt(mse)

print("MANUAL MODEL")
print("B0 =", b0)
print("B1 =", b1)
print("Equation: Y =", b0, "+", b1, "* X")
print("MSE =", mse)
print("RMSE =", rmse)

# Predict Gold price for 2025
gold_2025 = b0 + b1*2025

print("Gold price in 2025 =", gold_2025)
print("Gold price per gram =", gold_2025/10)


# Sklearn comparison
model = LinearRegression()
model.fit(x.reshape(-1, 1), y)

print("\nSKLEARN MODEL")
print("B0 =", model.intercept_)
print("B1 =", model.coef_[0])
print("Sklearn 2025 prediction =", model.predict([[2025]])[0])

MANUAL MODEL
B0 = -2084019.9875212705
B1 = 1048.101774572563
Equation: Y = -2084019.9875212705 + 1048.101774572563 * X
MSE = 52366665.82516636
RMSE = 7236.48159157241
Gold price in 2025 = 38386.10598816979
Gold price per gram = 3838.610598816979

SKLEARN MODEL
B0 = -2084019.98752127
B1 = 1048.1017745725628
Sklearn 2025 prediction = 38386.10598816979


### Question 2

In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# Load data
df = pd.read_csv("gold_silver.csv")

X = df[["Year", "SilverPrice_Rs_per_kg"]].values
y = df["GoldPrice_Rs_per_10g"].values

# Add column of 1s for intercept
X1 = np.column_stack((np.ones(len(X)), X))

# Manual Multiple Linear Regression
B = np.linalg.inv(X1.T @ X1) @ X1.T @ y

b0 = B[0]
b1 = B[1]
b2 = B[2]

# Prediction
yp = X1 @ B

# Error, MSE, RMSE
error = y - yp
mse = np.mean(error**2)
rmse = np.sqrt(mse)

print("MANUAL MODEL")
print("B0 =", b0)
print("B1 =", b1)
print("B2 =", b2)

print("Equation:")
print("Gold =", b0, "+", b1, "* Year +", b2, "* Silver")

print("MSE =", mse)
print("RMSE =", rmse)


# Sklearn comparison
model = LinearRegression()
model.fit(X, y)

print("\nSKLEARN MODEL")
print("B0 =", model.intercept_)
print("B1 =", model.coef_[0])
print("B2 =", model.coef_[1])

silver_model = LinearRegression()
silver_model.fit(df[["Year"]].values, df["SilverPrice_Rs_per_kg"].values)
silver_2024_per_kg = silver_model.predict([[2024]])[0]

gold_2024_per_10g = model.predict([[2024, silver_2024_per_kg]])[0]

print("\nPREDICTIONS FOR 2024")
print("Silver (Rs/kg):", silver_2024_per_kg, " -> Rs/gram:", silver_2024_per_kg/1000)
print("Gold (Rs/10g):", gold_2024_per_10g, " -> Rs/gram:", gold_2024_per_10g/10)

MANUAL MODEL
B0 = -361709.8524098362
B1 = 181.2224544793513
B2 = 0.6242174532701311
Equation:
Gold = -361709.8524098362 + 181.2224544793513 * Year + 0.6242174532701311 * Silver
MSE = 15467100.414901897
RMSE = 3932.8234660230933

SKLEARN MODEL
B0 = -361709.85241565824
B1 = 181.22245448212766
B2 = 0.6242174532686103

PREDICTIONS FOR 2024
Silver (Rs/kg): 51670.46930286568  -> Rs/gram: 51.670469302865676
Gold (Rs/10g): 37338.00421359687  -> Rs/gram: 3733.800421359687


### Additional

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import SGDRegressor

# Load data
df = pd.read_csv("gold_silver.csv")

x = df["Year"].values.astype(float)
y = df["GoldPrice_Rs_per_10g"].values.astype(float)

# Standardize X for SGD
mean_x = np.mean(x)
std_x = np.std(x)
xs = (x - mean_x) / std_x

# Start coefficients
b0 = 0
b1 = 0

lr = 0.01
epochs = 10000

# Manual SGD
for epoch in range(epochs):
    for i in range(len(xs)):
        yp = b0 + b1 * xs[i]
        error = yp - y[i]

        b0 = b0 - lr * error
        b1 = b1 - lr * error * xs[i]


## original_b1 = b1 / std_x (if using without standardizing year)
## original_b0 = b0 - b1 * mean_x / std_x

# Predictions
yp = b0 + b1 * xs

error = y - yp
mse = np.mean(error ** 2)
rmse = np.sqrt(mse)

print("GOLD - MANUAL SGD")
print("B0 =", b0)
print("B1 =", b1)
print("MSE =", mse)
print("RMSE =", rmse)


# Sklearn
model = SGDRegressor(
    max_iter=10000,
    learning_rate="constant",
    eta0=0.01,
    random_state=42
)

model.fit(xs.reshape(-1, 1), y)

print("\nGOLD - SKLEARN SGD")
print("B0 =", model.intercept_[0])
print("B1 =", model.coef_[0])


# Predict Gold for 2025
x2025 = (2025 - mean_x) / std_x

gold_2025 = b0 + b1 * x2025

print("\nGold price in 2025 =", gold_2025)
print("Gold price per gram =", gold_2025 / 10)


# SILVER PRICE USING SGD

y = df["SilverPrice_Rs_per_kg"].values.astype(float)

b0 = 0
b1 = 0

for epoch in range(epochs):
    for i in range(len(xs)):
        yp = b0 + b1 * xs[i]
        error = yp - y[i]

        b0 = b0 - lr * error
        b1 = b1 - lr * error * xs[i]

yp = b0 + b1 * xs

error = y - yp
mse = np.mean(error ** 2)
rmse = np.sqrt(mse)

print("\nSILVER - MANUAL SGD")
print("B0 =", b0)
print("B1 =", b1)
print("MSE =", mse)
print("RMSE =", rmse)


# Sklearn
model = SGDRegressor(
    max_iter=10000,
    learning_rate="constant",
    eta0=0.01,
    random_state=42
)

model.fit(xs.reshape(-1, 1), y)

print("\nSILVER - SKLEARN SGD")
print("B0 =", model.intercept_[0])
print("B1 =", model.coef_[0])


# Predict Silver for 2024
x2024 = (2024 - mean_x) / std_x

silver_2024 = b0 + b1 * x2024

print("\nSilver price in 2024 =", silver_2024)
print("Silver price per gram =", silver_2024 / 1000)

GOLD - MANUAL SGD
B0 = 13838.521231010323
B1 = 13661.29675629062
MSE = 53290023.66207379
RMSE = 7300.001620689805

GOLD - SKLEARN SGD
B0 = 13837.413563375549
B1 = 12743.257510273901

Gold price in 2025 = 40325.00179448789
Gold price per gram = 4032.5001794487894

SILVER - MANUAL SGD
B0 = 20387.41706336586
B1 = 17669.07104143574
MSE = 95400423.18072574
RMSE = 9767.314020790247

SILVER - SKLEARN SGD
B0 = 20567.199150110493
B1 = 16907.055394032188

Silver price in 2024 = 53186.42454461272
Silver price per gram = 53.18642454461272
